# Aula 04 — Análise e Preparação de Dados (CRISP-DM)

Este notebook é focado nas duas primeiras etapas de dados do **CRISP-DM**:

1. **Entendimento dos Dados** (Data Understanding)
2. **Preparação dos Dados** (Data Preparation)

A ideia é praticar, passo a passo, o que deve ser feito em cada etapa **antes** de treinar qualquer modelo. Depois de terminar este notebook, os dados preparados aqui poderiam alimentar a etapa de *Modelagem* (ver `Aula_04_Machine_Learning.ipynb`, onde praticamos isso com outra base de dados).

Ao longo do notebook você vai encontrar blocos assim:

> 🧠 **Faça você mesmo agora!**
> Uma tarefa curta para você resolver em aula, na célula de código logo abaixo.


#### Importação das bibliotecas básicas

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

pd.set_option('display.max_columns', None)

# Base de dados: Titanic

- Fonte (adaptado): Kaggle — [Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic)
- Carregada aqui diretamente pelo Seaborn (uma cópia da mesma base), então não precisa baixar nenhum arquivo.

Descrição dos principais atributos:

* **survived**: 0 = não sobreviveu, 1 = sobreviveu (nosso possível *target*)
* **pclass**: classe da passagem (1ª, 2ª, 3ª)
* **sex**: sexo do passageiro
* **age**: idade
* **sibsp**: nº de irmãos/cônjuges a bordo
* **parch**: nº de pais/filhos a bordo
* **fare**: valor pago pela passagem
* **embarked**: porto de embarque (C, Q, S)
* **deck**: convés da cabine (muitos valores ausentes)

Qual seria o objetivo de analisar essa base de dados? **Entender quais fatores estão associados à sobrevivência dos passageiros** — essa pergunta guia todas as decisões das próximas etapas.

In [ ]:
titanic = sns.load_dataset('titanic')
titanic

---
# Parte 1 — Entendimento dos Dados (Data Understanding)

### 1.1 Coleta e primeiro contato com os dados

Antes de qualquer análise, precisamos saber: quantos registros e colunas temos? Quais os tipos de cada coluna?

In [ ]:
titanic.shape

In [ ]:
titanic.head()

In [ ]:
titanic.info()

> 🧠 **Faça você mesmo agora!**
> 1. Use `.tail(10)` para ver os últimos 10 registros.
> 2. Use `.sample(5)` para ver 5 registros aleatórios.

In [ ]:
# Seu código aqui




### 1.2 Descrição estatística dos dados

`describe()` resume rapidamente as variáveis numéricas (média, desvio padrão, mínimo, máximo, quartis) e, com `include='object'`, também resume as categóricas.

In [ ]:
titanic.describe()

In [ ]:
titanic.describe(include='object')

> 🧠 **Faça você mesmo agora!**
> Responda com código:
> 1. Qual a idade **média** e a idade **máxima** dos passageiros?
> 2. Qual foi a tarifa (`fare`) mais alta paga por um passageiro?
> 3. Quantas classes (`pclass`) diferentes existem na base?

In [ ]:
# Seu código aqui




### 1.3 Exploração visual

Gráficos ajudam a enxergar padrões que tabelas escondem: distribuições, desbalanceamento de classes e relações entre variáveis.

In [ ]:
sns.countplot(x='survived', data=titanic)
plt.title('Quantidade de sobreviventes (0 = não, 1 = sim)')
plt.show()

In [ ]:
plt.hist(x=titanic['age'].dropna(), bins=30)
plt.title('Distribuição de idade')
plt.xlabel('age')
plt.show()

In [ ]:
plt.hist(x=titanic['fare'], bins=30)
plt.title('Distribuição da tarifa paga (fare)')
plt.xlabel('fare')
plt.show()

In [ ]:
numericas = titanic.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numericas.corr(), annot=True, cmap='coolwarm')
plt.title('Correlação entre variáveis numéricas')
plt.show()

> 🧠 **Faça você mesmo agora!**
> 1. Crie um `sns.countplot` de `survived` separado por `sex` (dica: use o parâmetro `hue`).
> 2. Crie um `sns.boxplot` de `age` separado por `survived`.
> 3. Escreva em um comentário: o que esses dois gráficos sugerem sobre quem tinha mais chance de sobreviver?

In [ ]:
# Seu código aqui




### 1.4 Verificação da qualidade dos dados

Nesta etapa procuramos problemas que vão exigir tratamento na *Preparação dos Dados*: valores ausentes, duplicados e possíveis outliers.

In [ ]:
titanic.isnull().sum()

In [ ]:
# Percentual de valores ausentes por coluna
(titanic.isnull().sum() / len(titanic) * 100).sort_values(ascending=False)

In [ ]:
titanic.duplicated().sum()

⚠️ Esse número é alto, mas cuidado: essa base **não tem um identificador único de passageiro** (como um `id`), então duas pessoas com os mesmos valores em todas as colunas (ex.: mesma classe, sexo, idade, tarifa) aparecem como "duplicadas" sem necessariamente ser um erro de coleta. Sempre investigue o *porquê* antes de simplesmente remover duplicados.

In [ ]:
sns.boxplot(x=titanic['fare'])
plt.title('Possíveis outliers em fare')
plt.show()

> 🧠 **Faça você mesmo agora!**
> 1. Qual coluna tem o maior percentual de valores ausentes? Ela ainda seria útil para o problema?
> 2. A coluna `age` tem valores ausentes — isso é grave para o objetivo da análise? Por quê?
> 3. Escreva, em um comentário, se você removeria ou tentaria imputar (preencher) os valores ausentes de cada coluna problemática, e por quê.

In [ ]:
# Seu código / comentários aqui




#### ✅ Checklist do Entendimento dos Dados

- [x] Sei quantas linhas e colunas tenho, e o tipo de cada uma
- [x] Conheço as estatísticas básicas (média, min, max, distribuição)
- [x] Visualizei as variáveis mais importantes e a relação delas com o problema (`survived`)
- [x] Sei quais colunas têm valores ausentes, duplicados ou outliers


---
# Parte 2 — Preparação dos Dados (Data Preparation)

Esta é, normalmente, a etapa que **mais consome tempo** em um projeto real de Machine Learning. Vamos transformar a base "crua" que exploramos na Parte 1 em uma base pronta para um algoritmo.

### 2.1 Seleção dos dados

Nem toda coluna é útil. Vamos remover colunas redundantes ou com informação demais faltando, com base no que descobrimos na Parte 1.

In [ ]:
# 'deck' tem muitos valores ausentes; 'embark_town' e 'alive' repetem informação
# que já existe em 'embarked' e 'survived'; 'class' repete 'pclass'.
titanic_prep = titanic.drop(columns=['deck', 'embark_town', 'alive', 'class'])
titanic_prep.head()

> 🧠 **Faça você mesmo agora!**
> Olhando as colunas restantes (`who`, `adult_male`, `alone`), existe alguma outra que você removeria por ser redundante? Justifique em um comentário e, se quiser, remova-a com `.drop(columns=[...])`.

In [ ]:
# Seu código aqui




### 2.2 Limpeza dos dados (valores ausentes)

Duas estratégias comuns: preencher (**imputar**) com uma medida central, ou remover o registro/coluna.

In [ ]:
# 'age': preenchemos com a mediana (mais robusta a outliers do que a média)
titanic_prep['age'] = titanic_prep['age'].fillna(titanic_prep['age'].median())

# 'embarked': só 2 valores ausentes, preenchemos com o valor mais frequente (moda)
titanic_prep['embarked'] = titanic_prep['embarked'].fillna(titanic_prep['embarked'].mode()[0])

titanic_prep.isnull().sum()

> 🧠 **Faça você mesmo agora!**
> Em vez da mediana geral, preencha `age` com a **mediana por classe** (`pclass`) — passageiros de classes diferentes tendem a ter idades diferentes. Dica: `groupby('pclass')['age'].transform('median')`.

In [ ]:
# Seu código aqui




### 2.3 Construção de novas variáveis (Feature Engineering)

Às vezes a informação mais útil não está em uma única coluna, mas em uma combinação delas.

In [ ]:
# Tamanho da família a bordo (irmãos/cônjuges + pais/filhos + a própria pessoa)
titanic_prep['family_size'] = titanic_prep['sibsp'] + titanic_prep['parch'] + 1

# A pessoa estava sozinha?
titanic_prep['is_alone'] = (titanic_prep['family_size'] == 1).astype(int)

titanic_prep[['sibsp', 'parch', 'family_size', 'is_alone']].head()

> 🧠 **Faça você mesmo agora!**
> Crie uma variável categórica `faixa_etaria` a partir de `age`, com as categorias `'crianca'` (até 12 anos), `'adulto'` (13 a 59) e `'idoso'` (60+). Dica: use `pd.cut(...)`.

In [ ]:
# Seu código aqui




### 2.4 Formatação dos dados (codificação e escalonamento)

A maioria dos algoritmos só entende números. Precisamos:
- **Codificar** variáveis categóricas (texto → número)
- **Escalonar** variáveis numéricas para que fiquem em faixas comparáveis

In [ ]:
# One-hot encoding das variáveis categóricas
titanic_prep = pd.get_dummies(titanic_prep, columns=['sex', 'embarked'], drop_first=True)
titanic_prep.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
titanic_prep[['age', 'fare']] = scaler.fit_transform(titanic_prep[['age', 'fare']])
titanic_prep[['age', 'fare']].describe()

> 🧠 **Faça você mesmo agora!**
> A coluna `who` (`man`/`woman`/`child`) ainda é texto. Aplique `pd.get_dummies` também nela (com `drop_first=True`) e confira o resultado com `.head()`.

In [ ]:
# Seu código aqui




### 2.5 Divisão em treino e teste

Por fim, separamos os previsores (`X`) do alvo (`y`, `survived`) e dividimos em treino/teste — deixando os dados prontos para a etapa de *Modelagem* do CRISP-DM.

In [ ]:
from sklearn.model_selection import train_test_split

X = titanic_prep.drop(columns=['survived'])
y = titanic_prep['survived']

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

X_treinamento.shape, X_teste.shape

> 🧠 **Faça você mesmo agora!**
> Refaça o `train_test_split` usando `test_size=0.30` e o parâmetro `stratify=y` (mantém a mesma proporção de sobreviventes/não sobreviventes nos dois conjuntos). Compare os tamanhos resultantes com os obtidos acima.

In [ ]:
# Seu código aqui




#### ✅ Checklist da Preparação dos Dados

- [x] Selecionei apenas as colunas relevantes para o problema
- [x] Tratei valores ausentes
- [x] Criei novas variáveis (feature engineering)
- [x] Codifiquei variáveis categóricas e escalonei variáveis numéricas
- [x] Separei os dados em treino e teste
